In [ ]:
import cv2
import numpy as np
import os

INPUT_FOLDER  = r"data/new_dataset/"
OUTPUT_FOLDER = r"data/removed_background/"


def extract_board(image_path, output_path):

In [ ]:
    img = cv2.imread(image_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    _, mask = cv2.threshold(blur, 80, 255, cv2.THRESH_BINARY_INV)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    row_sums = np.sum(mask, axis=1)
    col_sums = np.sum(mask, axis=0)

    threshold = 0.3 * mask.shape[1] * 255
    rows = np.where(row_sums > threshold)[0]
    cols = np.where(col_sums > threshold)[0]

    y1, y2 = rows[0], rows[-1]
    x1, x2 = cols[0], cols[-1]

    result = img[y1:y2, x1:x2]
    cv2.imwrite(output_path, result)


exts = {".jpg", ".jpeg", ".png", ".bmp"}
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for fname in os.listdir(INPUT_FOLDER):
    if os.path.splitext(fname)[1].lower() not in exts:
        continue
    inp = os.path.join(INPUT_FOLDER, fname)
    out = os.path.join(OUTPUT_FOLDER, fname)
    extract_board(inp, out)
    print(fname)

In [ ]:
import cv2
import numpy as np
import os
import glob


def extract_chessboard_grid(image_path, output_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    kernel = np.ones((5, 5), np.uint8)
    dilated = cv2.dilate(edges, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    img_area = img.shape[0] * img.shape[1]
    best_rect = None

    for contour in contours:
        area = cv2.contourArea(contour)

        if 0.5 * img_area < area < 0.98 * img_area:
            perimeter = cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, 0.02 * perimeter, True)
            if len(approx) == 4:
                x, y, w, h = cv2.boundingRect(approx)
                aspect_ratio = w / h
                if 0.95 <= aspect_ratio <= 1.05:
                    best_rect = (x, y, w, h)
                    break
    if best_rect:
        x, y, w, h = best_rect
        cropped = img[y:y + h, x:x + w]
    else:
        height, width = img.shape[:2]
        crop_y = int(height * 0.065)
        crop_x = int(width * 0.065)
        cropped = img[crop_y:height - crop_y, crop_x:width - crop_x]
    cv2.imwrite(output_path, cropped)


def process_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    extensions = ("*.jpg", "*.jpeg", "*.png", "*.bmp")

    image_files = []

    for ext in extensions:
        image_files.extend(glob.glob(os.path.join(input_dir, ext)))

    for image_path in image_files:
        filename = os.path.basename(image_path)
        output_path = os.path.join(output_dir, filename)
        extract_chessboard_grid(image_path, output_path)
        print(filename)


process_directory(
    "data/removed_background/",
    "data/extracted_boards/"
)